# Feature Engineering

> Сорочан Дмитрий, [@legenda0008](https://t.me/legenda0008)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer


SEED = 143

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


In [2]:
train_df = pd.read_csv('data/train.csv')

X_train = train_df.drop(columns=['id', 'target'])
y_train = train_df['target']

Дубликатов нет (см. [EDA](./EDA.ipynb)). Значения `-1` не удаляю: они есть и в `test`, а сам факт пропуска может быть и небесполезен.


In [3]:
def normalized_gini(y_true, y_pred_proba):  # в cross_val_score по умолчанию его нет, но он легко выражается из roc_auc
    return 2 * roc_auc_score(y_true, y_pred_proba) - 1


gini_scorer = make_scorer(
    normalized_gini,
    response_method="predict_proba",
)

### Feature engineering

Для грубой проверки наборов признаков беру `GaussianNB`: он быстрый и вообще не требует настройки гиперпараметров.

Категориальные признаки заэнкожу внутри пайплайна, остальные оставлю как есть, скейлер не нужен.


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
            make_column_selector(pattern='_cat$')
        ),
    ],
    remainder='passthrough',
    n_jobs=-1
)

feature_model = make_pipeline(
    preprocessor,
    GaussianNB()
)

scores = cross_val_score(feature_model, X_train, y_train, cv=cv, scoring=gini_scorer)

pd.DataFrame({
    'experiment': ['all_features'],
    'mean_gini': [scores.mean()],
    'std_gini': [scores.std()],
})


,experiment,mean_gini,std_gini
0,all_features,0.180504,0.009002


Отдельно проверю заполнение пропусков медианой по числовым признакам; а чтобы не потерять сам факт пропуска -- добавлю для них отдельные флаги:

In [5]:
numeric_columns = [
    column for column in X_train.columns
    if not column.endswith(('_cat', '_bin'))
]

preprocessor_with_imputation = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
            make_column_selector(pattern='_cat$')
        ),
        (
            'num',
            SimpleImputer(missing_values=-1, strategy='median', add_indicator=True),
            numeric_columns
        ),
    ],
    remainder='passthrough',
    n_jobs=-1
)

feature_model_with_imputation = make_pipeline(
    preprocessor_with_imputation,
    GaussianNB()
)

scores_with_imputation = cross_val_score(
    feature_model_with_imputation,
    X_train,
    y_train,
    cv=cv,
    scoring=gini_scorer
)

pd.DataFrame({
    'experiment': ['median_with_missing_flags'],
    'mean_gini': [scores_with_imputation.mean()],
    'std_gini': [scores_with_imputation.std()],
})


,experiment,mean_gini,std_gini
0,median_with_missing_flags,0.180869,0.008906


Медиана с флагами дала совсем небольшой прирост. Он заметно меньше разброса по фолдам, поэтому явного выигрыша здесь нет.


Удалю почти константы _(визуально нахожу по графикам в [EDA](./EDA.ipynb) такие признаки)_:

In [6]:
X_train_no_constants = X_train.drop(columns=['ps_ind_05_cat', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_car_04_cat', 'ps_car_07_cat', 'ps_car_10_cat'])

scores_no_constants = cross_val_score(feature_model, X_train_no_constants, y_train, cv=cv, scoring=gini_scorer)

pd.DataFrame({
    'experiment': ['no_constants'],
    'mean_gini': [scores_no_constants.mean()],
    'std_gini': [scores_no_constants.std()],
})


,experiment,mean_gini,std_gini
0,no_constants,0.158353,0.009391


Плохо.

Удалю меньшее их количество (прям совсем константы-константы):

In [7]:
X_train_no_constants_lite = X_train.drop(columns=['ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_car_10_cat'])

scores_no_constants_lite = cross_val_score(feature_model, X_train_no_constants_lite, y_train, cv=cv, scoring=gini_scorer)

pd.DataFrame({
    'experiment': ['no_constants_lite'],
    'mean_gini': [scores_no_constants_lite.mean()],
    'std_gini': [scores_no_constants_lite.std()],
})


,experiment,mean_gini,std_gini
0,no_constants_lite,0.180968,0.009682


Скор получился чуть выше, чем на полном наборе, но разница сильно меньше разброса по фолдам.


Удалю признаки с равномерным распределением:

In [8]:
X_train_no_uni = X_train.drop(columns=['ps_calc_01', 'ps_calc_02', 'ps_calc_03'])

scores_no_uni = cross_val_score(feature_model, X_train_no_uni, y_train, cv=cv, scoring=gini_scorer)

pd.DataFrame({
    'experiment': ['no_uniform'],
    'mean_gini': [scores_no_uni.mean()],
    'std_gini': [scores_no_uni.std()],
})


,experiment,mean_gini,std_gini
0,no_uniform,0.180488,0.009004


Удаление равномерных признаков практически ничего не изменило: результат чуть ниже исходного.


In [9]:
X_train_no_constants_lite_no_uni = X_train.drop(columns=['ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_car_10_cat', 'ps_calc_01', 'ps_calc_02', 'ps_calc_03'])

scores_no_constants_lite_no_uni = cross_val_score(feature_model, X_train_no_constants_lite_no_uni, y_train, cv=cv, scoring=gini_scorer)

pd.DataFrame({
    'experiment': ['no_constants_lite_no_uniform'],
    'mean_gini': [scores_no_constants_lite_no_uni.mean()],
    'std_gini': [scores_no_constants_lite_no_uni.std()],
})


,experiment,mean_gini,std_gini
0,no_constants_lite_no_uniform,0.180965,0.009676


Комбинация дала практически тот же результат, что и `no_constants_lite`: дополнительное удаление равномерных признаков ничего не добавило.


Можно было бы попробовать расширить признаковое пространство с помощью `PolynomialFeatures`, но на 57  признаках он быстро раздует матрицу, при этом не думаю, что это будет оправданно...

Логарифмы тоже добавлять не буду: больших хвостов у числовых признаков нет, контрить там нечего.

Вместо этого попробую уменьшить размерность пространства при помощи `PCA`.


In [10]:
components_list = [2, 5, 10, 20, 50]
pca_results = []

for n_components in components_list:
    pipeline = make_pipeline(
        preprocessor,
        StandardScaler(),
        PCA(n_components=n_components, random_state=SEED),
        GaussianNB()
    )

    scores_pca = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=gini_scorer
    )

    pca_results.append({
        'n_components': n_components,
        'mean_gini': scores_pca.mean(),
        'std_gini': scores_pca.std()
    })

pd.DataFrame(pca_results)


,n_components,mean_gini,std_gini
0,2,0.190694,0.008413
1,5,0.174324,0.009997
2,10,0.161356,0.012352
3,20,0.169854,0.011714
4,50,0.183132,0.011360


`PCA` зарешал: вариант с двумя компонентами дал лучший результат среди всех экспериментов _(не ожидал, честно говоря...)_. При увеличении числа компонент качество сначала падает, а на 50 компонентах восстанавливается лишь частично.


In [11]:
feature_results = pd.DataFrame([
    {
        'experiment': 'all_features',
        'n_features': X_train.shape[1],
        'mean_gini': scores.mean(),
        'std_gini': scores.std(),
    },
    {
        'experiment': 'median_with_missing_flags',
        'n_features': X_train.shape[1],
        'mean_gini': scores_with_imputation.mean(),
        'std_gini': scores_with_imputation.std(),
    },
    {
        'experiment': 'no_constants',
        'n_features': X_train_no_constants.shape[1],
        'mean_gini': scores_no_constants.mean(),
        'std_gini': scores_no_constants.std(),
    },
    {
        'experiment': 'no_constants_lite',
        'n_features': X_train_no_constants_lite.shape[1],
        'mean_gini': scores_no_constants_lite.mean(),
        'std_gini': scores_no_constants_lite.std(),
    },
    {
        'experiment': 'no_uniform',
        'n_features': X_train_no_uni.shape[1],
        'mean_gini': scores_no_uni.mean(),
        'std_gini': scores_no_uni.std(),
    },
    {
        'experiment': 'no_constants_lite_no_uniform',
        'n_features': X_train_no_constants_lite_no_uni.shape[1],
        'mean_gini': scores_no_constants_lite_no_uni.mean(),
        'std_gini': scores_no_constants_lite_no_uni.std(),
    },
    *[
        {
            'experiment': f"pca_{result['n_components']}",
            'n_features': result['n_components'],
            'mean_gini': result['mean_gini'],
            'std_gini': result['std_gini'],
        }
        for result in pca_results
    ],
]).sort_values('mean_gini', ascending=False).reset_index(drop=True)

feature_results


,experiment,n_features,mean_gini,std_gini
0,pca_2,2,0.190694,0.008413
1,pca_50,50,0.183132,0.011360
2,no_constants_lite,51,0.180968,0.009682
3,no_constants_lite_no_uniform,48,0.180965,0.009676
4,median_with_missing_flags,57,0.180869,0.008906
5,all_features,57,0.180504,0.009002
6,no_uniform,54,0.180488,0.009004
7,pca_5,5,0.174324,0.009997
8,pca_20,20,0.169854,0.011714
9,pca_10,10,0.161356,0.012352


Лучший результат в целом дал `PCA` с двумя компонентами.

Среди вариантов без понижения размерности лучшим оказался `no_constants_lite`, но его отрыв от полного набора сильно меньше разброса по фолдам. Для исходного пространства оставляю этот компактный вариант: **удаляю только совсем почти константные признаки**.


In [12]:
X_train = X_train_no_constants_lite

test_df = pd.read_csv('data/test.csv')
X_test = test_df.drop(columns=['id', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_14', 'ps_car_10_cat'])


Энкодинг категориальных признаков буду делать **внутри пайплайна** конкретной модели. Для ветки с `PCA` там же останутся скейлер и само понижение размерности, чтобы не было утечек.


In [13]:
pd.DataFrame({
    'dataset': ['train', 'test'],
    'rows': [X_train.shape[0], X_test.shape[0]],
    'features': [X_train.shape[1], X_test.shape[1]],
})


,dataset,rows,features
0,train,595212,51
1,test,892816,51


In [14]:
X_train.head()


,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_15,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
0,2,2,5,1,0,0,1,0,0,11,...,9,1,5,8,0,1,1,0,0,1
1,1,1,7,0,0,0,0,1,0,3,...,3,1,1,9,0,1,1,0,1,0
2,5,4,9,1,0,0,0,1,0,12,...,4,2,7,7,0,1,1,0,1,0
3,0,1,2,0,0,1,0,0,0,8,...,2,2,4,9,0,0,0,0,0,0
4,0,2,0,1,0,1,0,0,0,9,...,3,1,1,3,0,0,0,1,1,0
